# Linear Probing: System vs User Instruction Following

Extracts residual stream activations from both **TransformerLens** and **nnterp**, then trains a linear probe at each layer to predict whether the model follows the system prompt or the user prompt (Condition C of the dataset). Running both backends lets you compare numerical consistency and ergonomics.

In [1]:
# Dependencies are managed via uv (pyproject.toml at repo root).
# On a new instance, run from the repo root:
#   ./lambda-sync.sh <name>.sync.env setup
# Then source your config before launching Jupyter:
#   source <name>.sync.env && uv run jupyter lab

## 2. Imports

In [2]:
import json
import gc
import os
import time
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
import torch
from transformers import AutoTokenizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## 3. Configuration

In [12]:
import re

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

# Locate repo root by walking up until pyproject.toml is found
def _find_repo_root():
    p = Path.cwd()
    for candidate in [p, *p.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return p

REPO_ROOT = _find_repo_root()
DATA_DIR        = REPO_ROOT / "phase0_behavioral_analysis" / "data" / "results"
ACTIVATIONS_DIR = REPO_ROOT / "phase1_linear_probing" / "data" / "activations2"
REPORTS_DIR     = REPO_ROOT / "phase1_linear_probing" / "reports_roc"

ACTIVATIONS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# Auto-load *.sync.env from repo root so the kernel picks up HF_TOKEN
# without needing to source it in a terminal first.
def _load_sync_env(repo_root):
    pattern = re.compile(r'^export\s+(\w+)=(.*)')
    for env_file in sorted(repo_root.glob("*.sync.env")):
        with open(env_file) as f:
            for line in f:
                m = pattern.match(line.strip())
                if m:
                    key, val = m.group(1), m.group(2).strip('"\'')
                    os.environ.setdefault(key, val)
                    print("Set ", key)
        return env_file  # load first file found, stop
    return None

_env_file = _load_sync_env(REPO_ROOT)
if _env_file:
    print(f"Loaded env from: {_env_file.name}")

LABEL_MODE = "binary"  # "binary" (followed_system vs followed_user) or "one_vs_rest" (followed_system vs all)

TOKEN_POSITIONS = ["last_prompt"] #, "last_system", "last_user", "mean_all", "mean_system", "mean_user"]

N_CV_FOLDS = 5

# Scoring metric for all probes. Options: "roc_auc", "balanced_accuracy", "accuracy"
PROBE_METRIC = "roc_auc"

MAX_SAMPLES = None  # set to an int to cap samples (for fast debugging)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    print("WARNING: HF_TOKEN not set. Run: source <name>.sync.env")
else:
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

print(f"Device       : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(f"VRAM         : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"REPO_ROOT    : {REPO_ROOT}")
print(f"DATA_DIR     : {DATA_DIR}")
print(f"ACTIVATIONS  : {ACTIVATIONS_DIR}")
print(f"REPORTS      : {REPORTS_DIR}")
print(f"PROBE_METRIC : {PROBE_METRIC}")

Set  HF_TOKEN
Loaded env from: enrique.sync.env
Device       : cpu
REPO_ROOT    : /Users/enrique/system-user-circuits
DATA_DIR     : /Users/enrique/system-user-circuits/phase0_behavioral_analysis/data/results
ACTIVATIONS  : /Users/enrique/system-user-circuits/phase1_linear_probing/data/activations2
REPORTS      : /Users/enrique/system-user-circuits/phase1_linear_probing/reports_roc
PROBE_METRIC : roc_auc


## 4. Data Loading

In [13]:
def load_results(data_dir, model_name):
    safe_name = model_name.replace("/", "_")
    path = Path(data_dir) / f"{safe_name}_results.jsonl"
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return pd.DataFrame(records)


df_all = load_results(DATA_DIR, MODEL_NAME)
df = df_all[df_all["condition"] == "C"].copy()

if LABEL_MODE == "binary":
    df = df[df["label"].isin(["followed_system", "followed_user"])].copy()

df["y"] = (df["label"] == "followed_system").astype(int)

if MAX_SAMPLES is not None:
    df = df.sample(n=min(MAX_SAMPLES, len(df)), random_state=42)

df = df.reset_index(drop=True)

print(f"Condition C samples : {len(df)}")
print(f"followed_system     : {df['y'].sum()} ({df['y'].mean():.1%})")
print(f"other               : {(df['y'] == 0).sum()} ({(1-df['y'].mean()):.1%})")
print(f"\nLabel breakdown:\n{df['label'].value_counts()}")

Condition C samples : 3050
followed_system     : 552 (18.1%)
other               : 2498 (81.9%)

Label breakdown:
label
followed_user      2498
followed_system     552
Name: count, dtype: int64


## 5. Tokenizer & Prompt Utilities

In [14]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def build_formatted_prompt(system_text, user_text):
    messages = [
        {"role": "system", "content": system_text},
        {"role": "user", "content": user_text},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def _encode_len(text):
    return len(tokenizer.encode(text, add_special_tokens=False))


def find_token_positions(system_text, user_text):
    messages = [
        {"role": "system", "content": system_text},
        {"role": "user", "content": user_text},
    ]
    full_str = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    sys_str = tokenizer.apply_chat_template(
        [{"role": "system", "content": system_text}],
        tokenize=False,
        add_generation_prompt=False,
    )
    sys_user_str = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )

    n_full = _encode_len(full_str)
    n_sys = min(_encode_len(sys_str), n_full - 1)
    n_sys_user = min(_encode_len(sys_user_str), n_full - 1)

    n_sys = max(n_sys, 1)
    n_sys_user = max(n_sys_user, n_sys + 1)

    return {
        "last_prompt": n_full - 1,
        "last_system": n_sys - 1,
        "last_user": n_sys_user - 1,
        "mean_all": (0, n_full),
        "mean_system": (0, n_sys),
        "mean_user": (n_sys, n_sys_user),
    }

## 6. Precompute Prompts & Token Positions

In [15]:
formatted_prompts = []
position_maps = []
input_ids_list = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Preparing"):
    fp = build_formatted_prompt(row["system_prompt"], row["user_prompt"])
    pm = find_token_positions(row["system_prompt"], row["user_prompt"])
    ids = tokenizer(fp, return_tensors="pt", add_special_tokens=False).input_ids
    formatted_prompts.append(fp)
    position_maps.append(pm)
    input_ids_list.append(ids)

print(f"Prepared {len(formatted_prompts)} prompts")
print(f"Token count range: {min(t.shape[1] for t in input_ids_list)} – {max(t.shape[1] for t in input_ids_list)}")
print(f"Example positions : {position_maps[0]}")

Preparing:   0%|          | 0/3050 [00:00<?, ?it/s]

Prepared 3050 prompts
Token count range: 56 – 122
Example positions : {'last_prompt': 78, 'last_system': 48, 'last_user': 74, 'mean_all': (0, 79), 'mean_system': (0, 49), 'mean_user': (49, 75)}


## 7. Extraction Helpers

Shared utilities used by both backends.

In [16]:
def slice_activation(act_tensor, pos_val):
    if isinstance(pos_val, int):
        return act_tensor[0, pos_val, :].float().cpu().numpy()
    start, end = pos_val
    return act_tensor[0, start:end, :].float().mean(dim=0).cpu().numpy()


def unwrap_saved(proxy):
    return proxy.value if hasattr(proxy, "value") else proxy


def build_activation_array(buffers, n_samples, n_layers):
    return {
        pos: np.array(
            [[buffers[pos][layer][i] for layer in range(n_layers)] for i in range(n_samples)]
        )
        for pos in TOKEN_POSITIONS
    }


def save_activations(activations, path):
    np.savez_compressed(path, **activations)
    print(f"Saved: {path}")


def load_activations(path):
    loaded = np.load(path)
    return {k: loaded[k] for k in loaded.files}

## 8. TransformerLens: Load → Extract → Save → Cleanup

In [ ]:
from transformer_lens import HookedTransformer

model_tl = HookedTransformer.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device=DEVICE,
)
model_tl.eval()

N_LAYERS = model_tl.cfg.n_layers
D_MODEL = model_tl.cfg.d_model
print(f"Loaded via TL  |  layers={N_LAYERS}  d_model={D_MODEL}")

In [ ]:
buffers_tl = {pos: [[] for _ in range(N_LAYERS)] for pos in TOKEN_POSITIONS}

t0 = time.time()
for ids, pm in tqdm(zip(input_ids_list, position_maps), total=len(input_ids_list), desc="TL extract"):
    ids_gpu = ids.to(DEVICE)
    with torch.no_grad():
        _, cache = model_tl.run_with_cache(
            ids_gpu,
            prepend_bos=False,
            names_filter=lambda name: name.endswith("resid_post"),
        )
    for layer in range(N_LAYERS):
        act = cache["resid_post", layer]
        for pos_name in TOKEN_POSITIONS:
            buffers_tl[pos_name][layer].append(slice_activation(act, pm[pos_name]))
    del cache
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

tl_time = time.time() - t0
print(f"TL extraction: {tl_time:.1f}s  ({tl_time/len(input_ids_list):.2f}s/sample)")

activations_tl = build_activation_array(buffers_tl, len(input_ids_list), N_LAYERS)
del buffers_tl

safe = MODEL_NAME.replace("/", "_")
save_activations(activations_tl, ACTIVATIONS_DIR / f"act_tl_{safe}.npz")

print("Shapes:")
for pos, arr in activations_tl.items():
    print(f"  {pos}: {arr.shape}")

In [9]:
del model_tl
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    print(f"VRAM freed. Current usage: {torch.cuda.memory_allocated()/1e9:.2f} GB")

VRAM freed. Current usage: 0.04 GB


## 9. nnsight: Load → Extract → Save → Cleanup

Uses `nnsight.LanguageModel` with the explicit `tracer.invoke()` pattern.
See `NNSIGHT_NOTES.md` for API details.

In [ ]:
from nnsight import LanguageModel

# nnsight 0.5.x API notes (verified with probe_syntax_test.py):
#
# 1. model.trace(input_ids=ids) FAILS — kwargs-only means no Invoker is created,
#    so the body code runs outside the forward pass (interleaving=False).
#    Use either: model.trace(string_or_tensor)  (positional arg)
#            or: model.trace() + tracer.invoke(input_ids=ids)  (explicit invoker)
#
# 2. .save() returns torch.Tensor directly — no .value wrapper.
#    Tensors may have requires_grad=True, so use .detach() before .numpy().
#
# 3. With a single invoke, the batch dim is squeezed:
#    output[0] shape is (seq_len, d_model), NOT (1, seq_len, d_model).
#
# 4. nnterp.StandardizedTransformer fails to load Llama on nnsight 0.5.x
#    (FakeTensor incompatibility in check_model_renaming scan).
#    Use nnsight.LanguageModel directly instead.

model_nn = LanguageModel(
    MODEL_NAME,
    device_map="auto",
    dispatch=True,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
)

N_LAYERS_NN = len(list(model_nn.model.layers))
assert N_LAYERS_NN == N_LAYERS, f"Layer count mismatch: TL={N_LAYERS}, nnsight={N_LAYERS_NN}"
print(f"Loaded via nnsight LanguageModel  |  layers={N_LAYERS_NN}")

In [ ]:
buffers_nn = {pos: [[] for _ in range(N_LAYERS_NN)] for pos in TOKEN_POSITIONS}

t0 = time.time()
for ids, pm in tqdm(zip(input_ids_list, position_maps), total=len(input_ids_list), desc="nnsight extract"):
    ids_gpu = ids.to(DEVICE)

    saved = [None] * N_LAYERS_NN
    with model_nn.trace() as tracer:
        with tracer.invoke(input_ids=ids_gpu):
            for layer in range(N_LAYERS_NN):
                saved[layer] = model_nn.model.layers[layer].output[0].save()

    for layer in range(N_LAYERS_NN):
        hs = saved[layer]  # (seq_len, d_model) — batch dim squeezed
        if hs.dim() == 3:
            hs = hs[0]
        for pos_name in TOKEN_POSITIONS:
            pos_val = pm[pos_name]
            if isinstance(pos_val, int):
                vec = hs[pos_val, :].detach().float().cpu().numpy()
            else:
                start, end = pos_val
                vec = hs[start:end, :].detach().float().mean(0).cpu().numpy()
            buffers_nn[pos_name][layer].append(vec)

    if DEVICE == "cuda":
        torch.cuda.empty_cache()

nn_time = time.time() - t0
print(f"nnsight extraction: {nn_time:.1f}s  ({nn_time/len(input_ids_list):.2f}s/sample)")

activations_nn = build_activation_array(buffers_nn, len(input_ids_list), N_LAYERS)
del buffers_nn

safe = MODEL_NAME.replace("/", "_")
save_activations(activations_nn, ACTIVATIONS_DIR / f"act_nn_{safe}.npz")

if "tl_time" in globals():
    print(f"\nSpeed comparison:")
    print(f"  TransformerLens : {tl_time:.1f}s  ({tl_time/len(input_ids_list):.2f}s/sample)")
    print(f"  nnsight         : {nn_time:.1f}s  ({nn_time/len(input_ids_list):.2f}s/sample)")
    print(f"  ratio (TL/nnsight): {tl_time/nn_time:.2f}x")

In [ ]:
from accelerate.hooks import remove_hook_from_submodules
remove_hook_from_submodules(model_nn._model)
del model_nn
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    print(f"VRAM after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB")

## 10. Load Saved Activations

Run this cell if restarting after extraction to skip re-running the models.

In [8]:
safe = MODEL_NAME.replace("/", "_")
activations_tl = load_activations(ACTIVATIONS_DIR / f"act_tl_{safe}.npz")
N_LAYERS = activations_tl[TOKEN_POSITIONS[0]].shape[1]
print(f"N_LAYERS={N_LAYERS}")
for pos in TOKEN_POSITIONS:
    print(f"  TL  {pos}: {activations_tl[pos].shape}")

N_LAYERS=32
  TL  last_prompt: (3050, 32, 4096)
  TL  last_system: (3050, 32, 4096)
  TL  last_user: (3050, 32, 4096)
  TL  mean_all: (3050, 32, 4096)
  TL  mean_system: (3050, 32, 4096)
  TL  mean_user: (3050, 32, 4096)


In [ ]:
safe = MODEL_NAME.replace("/", "_")
activations_nn = load_activations(ACTIVATIONS_DIR / f"act_nn_{safe}.npz")
for pos in TOKEN_POSITIONS:
    print(f"  NN  {pos}: {activations_nn[pos].shape}")

## 11. Numerical Comparison: TL vs nnterp

Checks whether both backends produce numerically consistent activations. If TL reimplements the model differently from HuggingFace, cosine similarity will be < 1.

In [12]:
from sklearn.metrics.pairwise import cosine_similarity

print(f"{'position':<16} {'layer 0':>10} {'layer mid':>10} {'layer last':>10} {'mean all':>10}")
print("-" * 60)

mid = N_LAYERS // 2
for pos in TOKEN_POSITIONS:
    tl = activations_tl[pos]  # (n_samples, n_layers, d_model)
    nn = activations_nn[pos]

    def mean_cos(layer):
        a = tl[:, layer, :].astype(np.float32)
        b = nn[:, layer, :].astype(np.float32)
        sims = np.array([cosine_similarity(a[[i]], b[[i]])[0, 0] for i in range(len(a))])
        return sims.mean()

    c0 = mean_cos(0)
    cm = mean_cos(mid)
    cl = mean_cos(N_LAYERS - 1)
    ca = np.mean([mean_cos(l) for l in range(N_LAYERS)])
    print(f"{pos:<16} {c0:>10.4f} {cm:>10.4f} {cl:>10.4f} {ca:>10.4f}")

print("\n1.0 = identical, <1.0 = divergence between TL reimplementation and HF")

position            layer 0  layer mid layer last   mean all
------------------------------------------------------------
last_prompt          1.0000     1.0000     1.0000     1.0000
last_system          1.0000     1.0000     1.0000     1.0000
last_user            1.0000     1.0000     1.0000     1.0000
mean_all             1.0000     1.0000     1.0000     1.0000
mean_system          1.0000     1.0000     1.0000     1.0000
mean_user            1.0000     1.0000     1.0000     1.0000

1.0 = identical, <1.0 = divergence between TL reimplementation and HF


## 12. Linear Probing

In [17]:
y = df["y"].values

In [18]:
# Limit BLAS threads so cross_val_score n_jobs can parallelize folds effectively
# (otherwise each fit grabs all cores for tiny matmuls, causing contention)
import threadpoolctl
threadpoolctl.threadpool_limits(2, "blas")


def probe_all_positions(activations, n_folds=N_CV_FOLDS, metric=PROBE_METRIC):
    n_layers = activations[TOKEN_POSITIONS[0]].shape[1]
    cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    results = {}
    for pos_name in tqdm(TOKEN_POSITIONS, desc="Probing"):
        X = activations[pos_name]
        rows = []
        for layer in range(n_layers):
            X_layer = X[:, layer, :]
            pipe = Pipeline([
                ("scaler", StandardScaler()),
                ("clf", LogisticRegression(max_iter=1000, C=1.0, solver="lbfgs")),
            ])
            scores = cross_val_score(pipe, X_layer, y, cv=cv,
                                     scoring=metric, n_jobs=-1)
            rows.append({"layer": layer, "mean": scores.mean(), "std": scores.std()})
        results[pos_name] = pd.DataFrame(rows)
    return results

In [19]:
print("Probing TransformerLens activations...")
probe_results_tl = probe_all_positions(activations_tl)

print(f"\nPeak {PROBE_METRIC} — TransformerLens:")
for pos, df_res in probe_results_tl.items():
    best = df_res.loc[df_res["mean"].idxmax()]
    print(f"  {pos:<16} {best['mean']:.3f} ± {best['std']:.3f}  @ layer {int(best['layer'])}")

Probing TransformerLens activations...


Probing:   0%|          | 0/1 [00:00<?, ?it/s]


Peak roc_auc — TransformerLens:
  last_prompt      0.981 ± 0.005  @ layer 26


In [ ]:
print("\nProbing nnterp activations...")
probe_results_nn = probe_all_positions(activations_nn)

print(f"\nPeak {PROBE_METRIC} — nnterp:")
for pos, df_res in probe_results_nn.items():
    best = df_res.loc[df_res["mean"].idxmax()]
    print(f"  {pos:<16} {best['mean']:.3f} ± {best['std']:.3f}  @ layer {int(best['layer'])}")

## 12b. Control Probe (Permuted Labels)

Trains the identical probe on **shuffled labels** to establish a chance-level baseline.  
If the real probe accuracy ≫ control accuracy, the residual stream genuinely encodes the system-vs-user decision.  
Control is run on TL activations only (TL ≈ nnterp numerically, so one is sufficient).

In [17]:
N_PERMUTATIONS = 1   # number of shuffles to average over

rng = np.random.default_rng(0)

def probe_control(activations, n_permutations=N_PERMUTATIONS, n_folds=N_CV_FOLDS, metric=PROBE_METRIC):
    """Same probe, but labels are randomly permuted each time. Mean over permutations."""
    n_layers = activations[TOKEN_POSITIONS[0]].shape[1]
    cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    results = {}
    for pos_name in tqdm(TOKEN_POSITIONS, desc="Control probing"):
        X = activations[pos_name]
        layer_scores = np.zeros((n_permutations, n_layers))
        for p in range(n_permutations):
            y_perm = rng.permutation(y)
            for layer in range(n_layers):
                X_layer = X[:, layer, :]
                pipe = Pipeline([
                    ("scaler", StandardScaler()),
                    ("clf", LogisticRegression(max_iter=1000, C=1.0, solver="lbfgs")),
                ])
                scores = cross_val_score(pipe, X_layer, y_perm, cv=cv,
                                         scoring=metric, n_jobs=-1)
                layer_scores[p, layer] = scores.mean()
        rows = [
            {"layer": l,
             "mean": layer_scores[:, l].mean(),
             "std":  layer_scores[:, l].std()}
            for l in range(n_layers)
        ]
        results[pos_name] = pd.DataFrame(rows)
    return results


print("Running control probe (shuffled labels) on TL activations...")
probe_results_ctrl = probe_control(activations_tl)

print(f"\nControl probe peak {PROBE_METRIC} (should be ≈ 0.50):")
for pos, df_res in probe_results_ctrl.items():
    best = df_res.loc[df_res["mean"].idxmax()]
    real_best = probe_results_tl[pos].loc[probe_results_tl[pos]["mean"].idxmax()]
    gap = real_best["mean"] - best["mean"]
    print(f"  {pos:<16} ctrl={best['mean']:.3f}  real={real_best['mean']:.3f}  gap=+{gap:.3f}")

Running control probe (shuffled labels) on TL activations...


Control probing:   0%|          | 0/6 [00:00<?, ?it/s]


Control probe peak roc_auc (should be ≈ 0.50):
  last_prompt      ctrl=0.514  real=0.981  gap=+0.467
  last_system      ctrl=0.518  real=0.864  gap=+0.346
  last_user        ctrl=0.528  real=0.978  gap=+0.451
  mean_all         ctrl=0.526  real=0.978  gap=+0.452
  mean_system      ctrl=0.519  real=0.849  gap=+0.329
  mean_user        ctrl=0.497  real=0.977  gap=+0.480


## 12b. Control Probe (Metadata-Only)

A **metadata-only classifier** trained on surface features of the prompt — sequence length, system-segment length, user-segment length — **without looking at any activations**.

If this control reaches similar accuracy as the representation probe, the probe's signal could be explained by spurious correlations in input length rather than the model's internal representations.  
If the control stays near chance (≈ 0.50) while the real probe is high, the representations genuinely encode the system-vs-user decision.

In [20]:
import sys
sys.path.insert(0, str(REPO_ROOT / "phase1_linear_probing"))

from metadata_clf import (
    build_category_lists, build_metadata_features, get_feature_names,
    get_feature_groups, run_linear_control, run_boosted_control,
)

cats = build_category_lists(df)
for k, v in cats.items():
    n = len(v)
    preview = v[:5]
    print(f"{k:<18} ({n}): {preview}{'...' if n > 5 else ''}")

X_meta = build_metadata_features(df, position_maps, cats)
feature_names = get_feature_names(cats)
FEATURE_GROUPS = get_feature_groups(cats)
print(f"\nX_meta shape: {X_meta.shape}  features: {len(feature_names)}")

# ── Linear control (all features) ────────────────────────────────────────────
res_linear = run_linear_control(X_meta, y, n_folds=N_CV_FOLDS, metric=PROBE_METRIC)
ctrl_meta_acc = res_linear["mean"]
ctrl_meta_std = res_linear["std"]
print(f"\nLinear metadata control   : {ctrl_meta_acc:.3f} ± {ctrl_meta_std:.3f}  ({PROBE_METRIC})")

# ── Boosted control (categorical features only — no length features) ─────────
# Length features (token counts) are redundant with the categorical one-hots
# and leak information that trees can overfit to, inflating the control ceiling.
length_idx = FEATURE_GROUPS["length_feats"]
cat_mask = np.ones(X_meta.shape[1], dtype=bool)
cat_mask[length_idx] = False
X_meta_cat = X_meta[:, cat_mask]

feature_names_cat = [f for i, f in enumerate(feature_names) if cat_mask[i]]
print(f"Boosted features: {X_meta_cat.shape[1]} (dropped {len(length_idx)} length features)")

res_boosted = run_boosted_control(X_meta_cat, y, n_folds=N_CV_FOLDS, metric=PROBE_METRIC)
ctrl_boosted_acc = res_boosted["mean"]
ctrl_boosted_std = res_boosted["std"]
print(f"Boosted metadata control : {ctrl_boosted_acc:.3f} ± {ctrl_boosted_std:.3f}  ({PROBE_METRIC})")
print(f"Boost over linear        : +{ctrl_boosted_acc - ctrl_meta_acc:.3f}")

print(f"\nChance baseline          : 0.500")
print(f"\nReal probe peak (TL)  [{PROBE_METRIC}]:")
for pos, df_res in probe_results_tl.items():
    best = df_res.loc[df_res["mean"].idxmax()]
    gap_lin = best["mean"] - ctrl_meta_acc
    gap_bst = best["mean"] - ctrl_boosted_acc
    print(f"  {pos:<16} {best['mean']:.3f}  gap vs linear: {gap_lin:+.3f}  gap vs boosted: {gap_bst:+.3f}")

constraint_type    (8): ['capitalization', 'disclaimer', 'emoji', 'format', 'language']...
strength           (3): ['medium', 'strong', 'weak']
user_style         (4): ['authority', 'helpfulness', 'jailbreak', 'with_instruction']
task_id            (16): ['climate', 'exercise', 'french_revolution', 'gravity', 'internet']...

X_meta shape: (3050, 36)  features: 36

Linear metadata control   : 0.903 ± 0.013  (roc_auc)
Boosted features: 32 (dropped 4 length features)
Boosted metadata control : 0.977 ± 0.004  (roc_auc)
Boost over linear        : +0.074

Chance baseline          : 0.500

Real probe peak (TL)  [roc_auc]:
  last_prompt      0.981  gap vs linear: +0.078  gap vs boosted: +0.005


### Metadata Classifier — Feature Importance & Group Ablation

Three complementary views:
1. **Coefficient importance** — logistic regression weights on scaled features (fit once on all data). Magnitude = linear contribution; sign = which class the feature pushes toward.
2. **Linear group ablation** — leave-one-group-out CV with logistic regression. AUC drop vs baseline shows each group's contribution under a linear model.
3. **Boosted group ablation** — same leave-one-group-out protocol but with HistGradientBoosting (nested CV). Captures interaction effects (e.g. constraint_type × direction) that the linear model misses.

In [23]:
from metadata_clf import fit_boosted_importances, run_group_ablation

# ── 1. Linear coefficient importance (all features) ──────────────────────────
_scaler = StandardScaler().fit(X_meta)
_X_sc   = _scaler.transform(X_meta)
_clf    = LogisticRegression(max_iter=1000, C=1.0, solver="lbfgs").fit(_X_sc, y)
linear_coefs = _clf.coef_[0]

# ── 2. Boosted permutation importance (categorical only) ─────────────────────
bst_res = fit_boosted_importances(X_meta_cat, y, metric=PROBE_METRIC)
boosted_imps = bst_res["importances_mean"]
boosted_stds = bst_res["importances_std"]
print(f"Boosted best params: {bst_res['best_params']}")

# ── Plot: side-by-side coefficient / importance bars ─────────────────────────
from plotly.subplots import make_subplots

order_lin = np.argsort(np.abs(linear_coefs))[::-1]
order_bst = np.argsort(boosted_imps)[::-1]

fig_imp = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        "Linear (LogReg scaled coefficients, all features)",
        "Boosted (permutation importance, categorical only)",
    ],
    horizontal_spacing=0.08,
)

fig_imp.add_trace(go.Bar(
    x=[feature_names[i] for i in order_lin],
    y=linear_coefs[order_lin],
    marker_color=["#d62728" if c > 0 else "#1f77b4" for c in linear_coefs[order_lin]],
    showlegend=False,
), row=1, col=1)

fig_imp.add_trace(go.Bar(
    x=[feature_names_cat[i] for i in order_bst],
    y=boosted_imps[order_bst],
    error_y=dict(type="data", array=boosted_stds[order_bst], thickness=1.2, width=3),
    marker_color="#2ca02c",
    showlegend=False,
), row=1, col=2)

fig_imp.update_layout(
    title="Metadata Classifier — Feature Importance Comparison",
    xaxis_tickangle=-40, xaxis2_tickangle=-40,
    width=1300, height=500,
    template="plotly_white",
)
fig_imp.update_yaxes(title_text="Coefficient", row=1, col=1)
fig_imp.update_yaxes(title_text="Score drop when shuffled", row=1, col=2)
fig_imp.show()

# ── 3. Linear group ablation (all features) ──────────────────────────────────
print("\n── Linear group ablation ──")
df_abl_lin = run_group_ablation(
    X_meta, y, FEATURE_GROUPS,
    classifier="linear", n_folds=N_CV_FOLDS, metric=PROBE_METRIC,
    baseline_mean=ctrl_meta_acc, baseline_std=ctrl_meta_std,
)
print(df_abl_lin.to_string(index=False))

# ── 4. Boosted group ablation (categorical only) ─────────────────────────────
# Recompute groups without length features (indices shift)
FEATURE_GROUPS_CAT = {k: v for k, v in FEATURE_GROUPS.items() if k != "length_feats"}
n_dropped = len(FEATURE_GROUPS["length_feats"])
FEATURE_GROUPS_CAT = {
    k: [i - n_dropped for i in v]
    for k, v in FEATURE_GROUPS_CAT.items()
}

print("\n── Boosted group ablation (categorical only) ──")
df_abl_bst = run_group_ablation(
    X_meta_cat, y, FEATURE_GROUPS_CAT,
    classifier="boosted", n_folds=N_CV_FOLDS, metric=PROBE_METRIC,
    baseline_mean=ctrl_boosted_acc, baseline_std=ctrl_boosted_std,
)
print(df_abl_bst.to_string(index=False))

# ── Plot: side-by-side ablation bars ─────────────────────────────────────────
fig_abl = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        f"Linear ablation ({PROBE_METRIC})",
        f"Boosted ablation — categorical only ({PROBE_METRIC})",
    ],
    horizontal_spacing=0.08,
)

for col, (df_abl, baseline) in enumerate(
    [(df_abl_lin, ctrl_meta_acc), (df_abl_bst, ctrl_boosted_acc)], start=1
):
    for _, row in df_abl.iterrows():
        is_baseline = row["group_dropped"] == "none (baseline)"
        color = "#aec7e8" if is_baseline else (
            "#d62728" if row["drop"] > 0.02 else
            "#ff9896" if row["drop"] > 0 else "#1f77b4"
        )
        fig_abl.add_trace(go.Bar(
            x=[row["group_dropped"]],
            y=[row["mean"]],
            error_y=dict(type="data", array=[row["std"]], thickness=1.5, width=5),
            marker_color=color,
            showlegend=False,
            hovertemplate=(
                f"<b>{row['group_dropped']}</b><br>"
                f"{PROBE_METRIC}={row['mean']:.4f} ± {row['std']:.4f}<br>"
                f"drop={row['drop']:+.4f}<extra></extra>"
            ),
        ), row=1, col=col)
    fig_abl.add_hline(
        y=baseline, line_dash="dash", line_color="black", line_width=1.2,
        row=1, col=col,
    )

y_min = min(df_abl_lin["mean"].min(), df_abl_bst["mean"].min()) - 0.05
y_max = max(df_abl_lin["mean"].max(), df_abl_bst["mean"].max()) + 0.03
fig_abl.update_yaxes(range=[max(0.4, y_min), min(1.0, y_max)])
fig_abl.update_layout(
    title=(
        f"Metadata Classifier — Group Ablation Comparison ({PROBE_METRIC})<br>"
        "<sub>Each bar = model retrained WITHOUT that feature group. Drop = baseline − ablated.</sub>"
    ),
    width=1100, height=480,
    template="plotly_white",
    bargap=0.35,
)
fig_abl.show()

safe = MODEL_NAME.replace("/", "_")
fig_imp.write_html(str(REPORTS_DIR / f"metadata_importance_{safe}.html"))
fig_abl.write_html(str(REPORTS_DIR / f"metadata_ablation_{safe}.html"))
print(f"\nSaved: metadata_importance and metadata_ablation to {REPORTS_DIR}")

Boosted best params: {'l2_regularization': 0.0012087541473056963, 'learning_rate': 0.2708160864249968, 'max_leaf_nodes': 44, 'min_samples_leaf': 42}



── Linear group ablation ──
  group_dropped     mean      std      drop
none (baseline) 0.902852 0.013438  0.000000
   length_feats 0.903008 0.013376 -0.000156
constraint_type 0.836658 0.008324  0.066194
       strength 0.893890 0.015192  0.008963
     user_style 0.789955 0.018479  0.112897
        task_id 0.902977 0.015189 -0.000124
      direction 0.901182 0.014822  0.001670

── Boosted group ablation (categorical only) ──
  group_dropped     mean      std     drop
none (baseline) 0.976502 0.003960 0.000000
constraint_type 0.840514 0.003895 0.135988
       strength 0.927184 0.003810 0.049318
     user_style 0.831641 0.021703 0.144861
        task_id 0.961702 0.012760 0.014800
      direction 0.933081 0.009481 0.043421



Saved: metadata_importance and metadata_ablation to /Users/enrique/system-user-circuits/phase1_linear_probing/reports_roc


## 13. Visualization

Solid lines = TransformerLens, dashed lines = nnterp. Color = token position.

In [24]:
COLORS = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b"]

# Build backend list from what's available
_backends = []
if "probe_results_tl" in dir(): _backends.append(("TL", probe_results_tl, "solid"))
if "probe_results_nn" in dir(): _backends.append(("nnterp", probe_results_nn, "dash"))

fig = go.Figure()

for i, pos_name in enumerate(TOKEN_POSITIONS):
    color = COLORS[i % len(COLORS)]

    for bi, (backend_label, probe_results, dash) in enumerate(_backends):
        df_res = probe_results[pos_name]
        layers = df_res["layer"].values
        means  = df_res["mean"].values
        stds   = df_res["std"].values

        show_legend = bi == 0
        fig.add_trace(go.Scatter(
            x=layers, y=means,
            mode="lines",
            name=pos_name,
            legendgroup=pos_name,
            showlegend=show_legend,
            line=dict(color=color, width=2, dash=dash),
        ))

        if bi == 0:
            fig.add_trace(go.Scatter(
                x=np.concatenate([layers, layers[::-1]]),
                y=np.concatenate([means + stds, (means - stds)[::-1]]),
                fill="toself", fillcolor=color, opacity=0.10,
                line=dict(color="rgba(0,0,0,0)"),
                legendgroup=pos_name, showlegend=False, hoverinfo="skip",
            ))

# Metadata control lines
fig.add_hline(
    y=ctrl_meta_acc,
    line_dash="dashdot", line_color="black", line_width=1.5,
    annotation_text=f"linear ctrl ({ctrl_meta_acc:.2f})",
    annotation_position="top left",
)
if "ctrl_boosted_acc" in dir():
    fig.add_hline(
        y=ctrl_boosted_acc,
        line_dash="dash", line_color="darkred", line_width=1.5,
        annotation_text=f"boosted ctrl ({ctrl_boosted_acc:.2f})",
        annotation_position="top left",
    )

# Chance line
fig.add_hline(
    y=0.5, line_dash="dot", line_color="gray",
    annotation_text="chance (0.50)", annotation_position="bottom right",
)

_backend_names = [b[0] for b in _backends]
_backend_desc = ", ".join(_backend_names) if len(_backend_names) > 1 else _backend_names[0]
label_str = "followed_user" if LABEL_MODE == "binary" else "other"
fig.update_layout(
    title=(
        f"Linear Probe: followed_system vs {label_str}<br>"
        f"<sub>{MODEL_NAME} | backends: {_backend_desc} | {N_CV_FOLDS}-fold CV | metric={PROBE_METRIC}</sub>"
    ),
    xaxis_title="Layer",
    yaxis_title=PROBE_METRIC.replace("_", " ").title(),
    yaxis=dict(range=[0.4, 1.02]),
    legend_title="Token Position",
    width=1100,
    height=580,
    template="plotly_white",
    hovermode="x unified",
)

fig.show()

safe = MODEL_NAME.replace("/", "_")
out = REPORTS_DIR / f"probe_{safe}_{LABEL_MODE}_comparison.html"
fig.write_html(str(out))
print(f"Saved: {out}")

Saved: /Users/enrique/system-user-circuits/phase1_linear_probing/reports_roc/probe_meta-llama_Llama-3.1-8B-Instruct_binary_comparison.html


## 14. Summary Table

In [25]:
_all_backends = []
if "probe_results_tl" in dir(): _all_backends.append(("TL", probe_results_tl))
if "probe_results_nn" in dir(): _all_backends.append(("nnterp", probe_results_nn))

rows = []
for pos in TOKEN_POSITIONS:
    for backend, probe_results in _all_backends:
        df_res = probe_results[pos]
        best = df_res.loc[df_res["mean"].idxmax()]
        row_dict = {
            "backend": backend,
            "token_position": pos,
            f"peak_{PROBE_METRIC}": round(best["mean"], 4),
            "std": round(best["std"], 4),
            "peak_layer": int(best["layer"]),
            "peak_layer_%": f"{int(best['layer']) / N_LAYERS * 100:.0f}%",
            "linear_ctrl": round(ctrl_meta_acc, 4),
            "gap_vs_linear": round(best["mean"] - ctrl_meta_acc, 4),
        }
        if "ctrl_boosted_acc" in dir():
            row_dict["boosted_ctrl"] = round(ctrl_boosted_acc, 4)
            row_dict["gap_vs_boosted"] = round(best["mean"] - ctrl_boosted_acc, 4)
        rows.append(row_dict)

summary = (
    pd.DataFrame(rows)
    .sort_values(["token_position", "backend"])
    .reset_index(drop=True)
)
print(summary.to_string(index=False))

backend token_position  peak_roc_auc    std  peak_layer peak_layer_%  linear_ctrl  gap_vs_linear  boosted_ctrl  gap_vs_boosted
     TL    last_prompt        0.9811 0.0046          26          81%       0.9029         0.0782        0.9765          0.0046


In [26]:
import plotly.graph_objects as go

fig2 = go.Figure()

_backends2 = []
if "probe_results_tl" in dir(): _backends2.append(("TL", probe_results_tl, "solid", 2.5, "#1f77b4"))
if "probe_results_nn" in dir(): _backends2.append(("nnterp", probe_results_nn, "dash", 2.0, "#4a9fd4"))

for backend_label, probe_results, dash, width, color in _backends2:
    df_res = probe_results["last_prompt"]
    layers = df_res["layer"].values
    means  = df_res["mean"].values
    stds   = df_res["std"].values

    fig2.add_trace(go.Scatter(
        x=layers, y=means,
        mode="lines",
        name=f"last_prompt ({backend_label})",
        line=dict(color=color, width=width, dash=dash),
    ))

    # CI band for first backend only
    if backend_label == _backends2[0][0]:
        fig2.add_trace(go.Scatter(
            x=np.concatenate([layers, layers[::-1]]),
            y=np.concatenate([means + stds, (means - stds)[::-1]]),
            fill="toself", fillcolor=color, opacity=0.12,
            line=dict(color="rgba(0,0,0,0)"),
            showlegend=False, hoverinfo="skip",
        ))

# Metadata control lines
fig2.add_hline(
    y=ctrl_meta_acc,
    line_dash="dashdot", line_color="crimson", line_width=1.5,
    annotation_text=f"linear ctrl ({ctrl_meta_acc:.3f})",
    annotation_position="top right",
    annotation_font_color="crimson",
)
if "ctrl_boosted_acc" in dir():
    fig2.add_hline(
        y=ctrl_boosted_acc,
        line_dash="dash", line_color="darkred", line_width=2,
        annotation_text=f"boosted ctrl ({ctrl_boosted_acc:.3f})",
        annotation_position="bottom right",
        annotation_font_color="darkred",
    )

# Chance line
fig2.add_hline(
    y=0.5, line_dash="dot", line_color="gray", line_width=1,
    annotation_text="chance (0.50)", annotation_position="bottom right",
)

fig2.update_layout(
    title=(
        f"Linear Probe — last_prompt token<br>"
        f"<sub>{MODEL_NAME} | {N_CV_FOLDS}-fold CV | metric={PROBE_METRIC}</sub>"
    ),
    xaxis_title="Layer",
    yaxis_title=PROBE_METRIC.replace("_", " ").title(),
    yaxis=dict(range=[0.4, 1.02]),
    legend=dict(x=0.02, y=0.05),
    width=800, height=480,
    template="plotly_white",
)

fig2.show()

safe = MODEL_NAME.replace("/", "_")
csv_path = REPORTS_DIR / f"probe_{safe}_summary.csv"
summary.to_csv(str(csv_path), index=False)
print(f"Saved: {csv_path}")

Saved: /Users/enrique/system-user-circuits/phase1_linear_probing/reports_roc/probe_meta-llama_Llama-3.1-8B-Instruct_summary.csv


In [27]:
df.groupby(["constraint_type"])["y"].mean()

constraint_type
capitalization    0.059896
disclaimer        0.197917
emoji             0.148438
format            0.348285
language          0.184896
list_format       0.361979
self_reference    0.002604
starting_word     0.144414
Name: y, dtype: float64

## Use "format" only

In [26]:
df_fmt = df[df["constraint_type"] == "format"].copy().reset_index(drop=True)
y_fmt = df_fmt["y"].values

print(f"Format-only samples: {len(df_fmt)}")
print(f"  followed_system: {y_fmt.sum()} ({y_fmt.mean():.1%})")
print(f"  followed_user:   {(y_fmt == 0).sum()} ({1 - y_fmt.mean():.1%})")

fmt_idx = df[df["constraint_type"] == "format"].index.values

activations_fmt = {}
for pos in TOKEN_POSITIONS:
    activations_fmt[pos] = activations_tl[pos][fmt_idx]

print(f"\nActivation shape check: {activations_fmt['last_prompt'].shape}")

Format-only samples: 379
  followed_system: 132 (34.8%)
  followed_user:   247 (65.2%)

Activation shape check: (379, 32, 4096)


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

n_layers = activations_fmt["last_prompt"].shape[1]

n_folds_fmt = 3
cv_fmt = StratifiedKFold(n_splits=n_folds_fmt, shuffle=True, random_state=42)

pos_name = "last_prompt"
X_pos = activations_fmt[pos_name]
rows_fmt = []
for layer in range(n_layers):
    X_layer = X_pos[:, layer, :]
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, C=1.0, solver="lbfgs", tol=1e-3)),
    ])
    scores = cross_val_score(pipe, X_layer, y_fmt, cv=cv_fmt,
                             scoring=PROBE_METRIC, n_jobs=-1)
    rows_fmt.append({"layer": layer, "mean": scores.mean(), "std": scores.std()})

probe_fmt = pd.DataFrame(rows_fmt)
best = probe_fmt.loc[probe_fmt["mean"].idxmax()]
print(f"Format-only probe (last_prompt) [{PROBE_METRIC}]:")
print(f"  Peak: {best['mean']:.3f} ± {best['std']:.3f} @ layer {int(best['layer'])}")

In [ ]:
ALL_SYS_STRENGTHS = sorted(df_fmt["strength"].unique())
ALL_USR_STYLES    = sorted(df_fmt["user_style"].unique())

def one_hot(val, categories):
    return [int(val == c) for c in categories]

# Rebuild position maps for format-only samples
position_maps_fmt = [position_maps[i] for i in fmt_idx]

X_meta_fmt = np.array([
    [
        pm["last_prompt"] + 1,
        pm["mean_system"][1],
        pm["mean_user"][1] - pm["mean_user"][0],
        pm["mean_user"][0],
        *one_hot(row["strength"],   ALL_SYS_STRENGTHS),
        *one_hot(row["user_style"], ALL_USR_STYLES),
        int(row["direction"] == "b_to_a"),
    ]
    for pm, (_, row) in zip(position_maps_fmt, df_fmt.iterrows())
], dtype=np.float32)

pipe_meta_fmt = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, C=1.0, solver="lbfgs")),
])
meta_scores_fmt = cross_val_score(pipe_meta_fmt, X_meta_fmt, y_fmt,
                                   cv=cv_fmt, scoring=PROBE_METRIC)

ctrl_fmt = meta_scores_fmt.mean()
ctrl_fmt_std = meta_scores_fmt.std()

gap = best["mean"] - ctrl_fmt

print(f"Format-only metadata control [{PROBE_METRIC}]: {ctrl_fmt:.3f} ± {ctrl_fmt_std:.3f}")
print(f"Format-only probe peak:                        {best['mean']:.3f} ± {best['std']:.3f}")
print(f"Gap:                                           +{gap:.3f}")

In [ ]:
N_PERMUTATIONS = 1
rng = np.random.default_rng(0)

perm_accs = []
X_best_layer = activations_fmt[pos_name][:, int(best["layer"]), :]
for _ in range(N_PERMUTATIONS):
    y_shuf = rng.permutation(y_fmt)
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=200, C=1.0, solver="saga")),
    ])
    scores = cross_val_score(pipe, X_best_layer, y_shuf, cv=cv_fmt,
                             scoring=PROBE_METRIC, n_jobs=-1)
    perm_accs.append(scores.mean())

perm_mean = np.mean(perm_accs)
print(f"Permuted-label baseline (layer {int(best['layer'])}) [{PROBE_METRIC}]: {perm_mean:.3f}")
print(f"Real probe at same layer:                                              {best['mean']:.3f}")
print(f"Gap vs permuted:                                                      +{best['mean'] - perm_mean:.3f}")

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

layers = probe_fmt["layer"].values
means  = probe_fmt["mean"].values
stds   = probe_fmt["std"].values

# Probe line
fig.add_trace(go.Scatter(
    x=layers, y=means, mode="lines",
    name="last_prompt (TL)",
    line=dict(color="#1f77b4", width=2.5),
))

# CI band
fig.add_trace(go.Scatter(
    x=np.concatenate([layers, layers[::-1]]),
    y=np.concatenate([means + stds, (means - stds)[::-1]]),
    fill="toself", fillcolor="#1f77b4", opacity=0.12,
    line=dict(color="rgba(0,0,0,0)"),
    showlegend=False, hoverinfo="skip",
))

# Metadata control
fig.add_hline(
    y=ctrl_fmt, line_dash="dashdot", line_color="crimson", line_width=2,
    annotation_text=f"metadata ctrl ({ctrl_fmt:.3f})",
    annotation_position="top right",
    annotation_font_color="crimson",
)

# Permuted baseline
fig.add_hline(
    y=perm_mean, line_dash="dot", line_color="gray", line_width=1.5,
    annotation_text=f"permuted labels ({perm_mean:.3f})",
    annotation_position="bottom right",
)

# Chance
fig.add_hline(
    y=0.5, line_dash="dot", line_color="lightgray", line_width=1,
    annotation_text="chance (0.50)", annotation_position="bottom right",
)

fig.update_layout(
    title=(
        f"Linear Probe — FORMAT constraint only (last_prompt)<br>"
        f"<sub>{MODEL_NAME} | {n_folds_fmt}-fold CV | n={len(df_fmt)} samples | metric={PROBE_METRIC}</sub>"
    ),
    xaxis_title="Layer",
    yaxis_title=PROBE_METRIC.replace("_", " ").title(),
    yaxis=dict(range=[0.3, 1.05]),
    width=900, height=500,
    template="plotly_white",
)

fig.show()

safe = MODEL_NAME.replace("/", "_")
out_path = REPORTS_DIR / f"probe_format_only_{safe}.html"
fig.write_html(str(out_path))
print(f"Saved: {out_path}")

In [32]:
print("=" * 60)
print("FORMAT-ONLY")
print("=" * 60)
print(f"Samples:              {len(df_fmt)}")
print(f"SCR (system win rate): {y_fmt.mean():.1%}")
print(f"")
print(f"Probe peak:            {best['mean']:.3f} ± {best['std']:.3f} @ layer {int(best['layer'])}")
print(f"Metadata control:      {ctrl_fmt:.3f} ± {ctrl_fmt_std:.3f}")
print(f"Permuted control:      {perm_mean:.3f}")
print(f"")
print(f"Gap (probe - metadata): +{best['mean'] - ctrl_fmt:.3f}")
print(f"Gap (probe - permuted): +{best['mean'] - perm_mean:.3f}")
print(f"")
if best["mean"] - ctrl_fmt > 0.05:
    print("Probe shows signal ABOVE metadata. Some activation-level info exists.")
else:
    print("Probe does NOT clearly beat metadata. Signal may be surface features.")
if best["mean"] - perm_mean > 0.1:
    print("Probe clearly above chance (permuted). Labels are learnable.")
else:
    print("Probe barely above permuted baseline. Weak or no signal.")
print("=" * 60)

FORMAT-ONLY
Samples:              379
SCR (system win rate): 34.8%

Probe peak:            0.880 ± 0.007 @ layer 28
Metadata control:      0.876 ± 0.006
Permuted control:      0.491

Gap (probe - metadata): +0.004
Gap (probe - permuted): +0.389

Probe does NOT clearly beat metadata. Signal may be surface features.
Probe clearly above chance (permuted). Labels are learnable.
